# Phase 3 Step 6 — One-Time November Lockbox Evaluation

November 30 was preserved while the target, 16 features, ET-2 configuration, uncalibrated score semantics, and top-10% policy were frozen in Steps 1–5. This is the first and only planned final evaluation. Results may assess generalization but cannot retune any component.

> **THE NOVEMBER 30 RETURN-RISK LOCKBOX IS CONSUMED BY THIS NOTEBOOK.**

## Frozen system and development evidence

Positive class 1 means zero baskets in `(T,T+28d]`, not churn. ET-2 is trained on April–August, unchanged. Development evidence is PR-AUC 0.4827, ROC-AUC 0.8576, Brier 0.0916, and log loss 0.2944. The primary policy is deterministic top 10%; top 5% and 20% are descriptive alternatives. No fixed threshold or calibrator exists.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import numpy as np,pandas as pd,pyreadr
from scipy.special import expit
from sklearn.metrics import average_precision_score,roc_auc_score,brier_score_loss,log_loss
from marketmind.return_risk.cohorts import build_labeled_cohort
from marketmind.return_risk.features import build_candidate_features,baseline_features,BASELINE_FEATURE_ORDER
from marketmind.return_risk.models import TRAINING_SNAPSHOTS,LOCKBOX_SNAPSHOT,fit_extra_trees_candidate,EXTRA_TREES_CANDIDATE_PARAMETERS
from marketmind.return_risk.policy import freeze_lockbox_scores
from marketmind.return_risk.evaluation import score_metrics
from marketmind.segmentation.bundle import load_bundle
from marketmind.segmentation.assignment import assign_from_transactions
pd.set_option('display.max_columns',40)
RAW=ROOT/'data'/'raw'/'complete_journey';ART=ROOT/'reports'/'return_risk'/'artifacts';ART.mkdir(parents=True,exist_ok=True)
assert EXTRA_TREES_CANDIDATE_PARAMETERS['ET-2']=={'n_estimators':300,'max_depth':12,'min_samples_leaf':10,'max_features':'sqrt','class_weight':None,'n_jobs':-1,'random_state':42}
assert LOCKBOX_SNAPSHOT==pd.Timestamp('2017-11-30 23:59:59')

## Final research training protocol

Reconstruct April–August exactly. No September/October rows are added before lockbox evaluation, matching the predeclared research protocol.

In [2]:
transactions=pyreadr.read_r(str(RAW/'transactions.rds'))[None];products=pyreadr.read_r(str(RAW/'products.rda'))['products'];transactions.transaction_timestamp=pd.to_datetime(transactions.transaction_timestamp)
train_frames=[]
for snapshot in TRAINING_SNAPSHOTS:
    X=baseline_features(build_candidate_features(transactions,products,snapshot));y=build_labeled_cohort(transactions,snapshot,28,'2017-12-31 23:59:59')[['return_risk_target']]
    z=X.join(y,validate='one_to_one').reset_index();z['snapshot_at']=snapshot;train_frames.append(z)
train=pd.concat(train_frames,ignore_index=True);assert len(train)==10223 and train.snapshot_at.max()==TRAINING_SNAPSHOTS[-1]
X_train=train.loc[:,BASELINE_FEATURE_ORDER];y_train=train.return_risk_target
model=fit_extra_trees_candidate('ET-2',X_train,y_train,train.snapshot_at)

## Feature cutoff and score-before-outcome freeze

The next cell builds November features using transactions at or before November 30, generates scores, ranks, and flags, and persists an outcome-free artifact. **No December target is constructed in or before this cell.**

In [3]:
november_features=baseline_features(build_candidate_features(transactions,products,LOCKBOX_SNAPSHOT));assert tuple(november_features.columns)==BASELINE_FEATURE_ORDER and len(november_features)==2322
frozen_scores=freeze_lockbox_scores(november_features,model,snapshot_at=LOCKBOX_SNAPSHOT)
PREOUTCOME=ART/'november_lockbox_scores_preoutcome.csv';frozen_scores.to_csv(PREOUTCOME,index=False)
assert 'return_risk_target' not in frozen_scores and frozen_scores.household_id.nunique()==2322
frozen_scores.head()

,household_id,snapshot_date,risk_score,risk_rank,top_05_flag,top_10_flag,top_20_flag
0,1538,2017-11-30 23:59:59,0.637629,1,True,True,True
1,252,2017-11-30 23:59:59,0.614090,2,True,True,True
2,1262,2017-11-30 23:59:59,0.595264,3,True,True,True
3,1343,2017-11-30 23:59:59,0.594752,4,True,True,True
4,2383,2017-11-30 23:59:59,0.584021,5,True,True,True


## First and only target access

Only after score persistence, construct the complete `(2017-11-30 23:59:59, 2017-12-28 23:59:59]` target. The cohort utility rejects right-censoring and counts distinct baskets.

In [4]:
lockbox_labels=build_labeled_cohort(transactions,LOCKBOX_SNAPSHOT,28,'2017-12-31 23:59:59')[['return_risk_target','future_baskets','outcome_end']].reset_index();assert lockbox_labels.outcome_end.eq(pd.Timestamp('2017-12-28 23:59:59')).all()
lockbox=frozen_scores.merge(lockbox_labels,on='household_id',how='left',validate='one_to_one');assert lockbox.return_risk_target.notna().all() and len(lockbox)==2322
lockbox.return_risk_target.agg(['count','sum','mean'])

count    2322.000000
sum       341.000000
mean        0.146856
Name: return_risk_target, dtype: float64

## Lockbox metrics and frozen behavioral baselines

Recency uses the April–August empirical rank definition. The heuristic uses the frozen equal-weight training-standardized recency minus log-frequency rule. They provide context only and cannot alter ET-2.

In [5]:
y=lockbox.return_risk_target.astype(int).to_numpy();et_score=lockbox.risk_score.to_numpy();recency_sorted=np.sort(train.recency_days.to_numpy());recency_score=np.searchsorted(recency_sorted,november_features.loc[lockbox.household_id,'recency_days'].to_numpy(),side='right')/len(recency_sorted)
lf=np.log1p(train.basket_frequency_lifetime);rm,rs=train.recency_days.mean(),train.recency_days.std(ddof=0);fm,fs=lf.mean(),lf.std(ddof=0);ordered=november_features.loc[lockbox.household_id];heuristic_score=expit((ordered.recency_days-rm)/rs-(np.log1p(ordered.basket_frequency_lifetime)-fm)/fs)
metric_rows=[]
for name,score in [('ET-2',et_score),('Recency',recency_score),('Heuristic',heuristic_score)]:
    metric_rows.append({'model':name,**score_metrics(y,score)})
metrics=pd.DataFrame(metric_rows);metrics.to_csv(ART/'november_lockbox_metrics.csv',index=False);display(metrics.drop(columns='confusion_matrix').round(5))

,model,n,prevalence,pr_auc,roc_auc,log_loss,brier,precision_at_0_5,recall_at_0_5,f1_at_0_5
0,ET-2,2322,0.14686,0.49991,0.86909,0.29954,0.09528,0.65000,0.07625,0.13648
1,Recency,2322,0.14686,0.45907,0.81359,0.95886,0.30954,0.22374,0.91202,0.35933
2,Heuristic,2322,0.14686,0.49938,0.86121,0.62830,0.17966,0.36122,0.83578,0.50442


## Frozen top-capacity policy

Flags were frozen before outcomes. Top 10% is primary; top 5% and 20% remain descriptive. Lift is flagged precision divided by lockbox prevalence.

In [6]:
capacity=[]
for column,label in [('top_05_flag','Top 5%'),('top_10_flag','Top 10%'),('top_20_flag','Top 20%')]:
    flag=lockbox[column].astype(bool);caught=int(lockbox.loc[flag,'return_risk_target'].sum());precision=caught/flag.sum();capacity.append({'policy':label,'flagged':int(flag.sum()),'positives_captured':caught,'precision':precision,'recall':caught/y.sum(),'prevalence':y.mean(),'lift':precision/y.mean()})
capacity=pd.DataFrame(capacity);capacity.to_csv(ART/'november_lockbox_capacity.csv',index=False);display(capacity.round(5))

,policy,flagged,positives_captured,precision,recall,prevalence,lift
0,Top 5%,117,66,0.56410,0.19355,0.14686,3.84119
1,Top 10%,233,129,0.55365,0.37830,0.14686,3.77000
2,Top 20%,465,216,0.46452,0.63343,0.14686,3.16307


## Development-to-lockbox generalization gap

In [7]:
development={'pr_auc':0.4826507400276322,'roc_auc':0.8576020491803279,'brier':0.09162544984904186,'log_loss':0.2944295581890584};et=metrics.set_index('model').loc['ET-2'];gaps=[]
for metric in development:
    lock=float(et[metric]);dev=development[metric];gaps.append({'metric':metric,'development':dev,'lockbox':lock,'absolute_gap':lock-dev,'relative_gap_pct':(lock/dev-1)*100})
gaps=pd.DataFrame(gaps);gaps.to_csv(ART/'november_generalization_gap.csv',index=False);display(gaps.round(5))

,metric,development,lockbox,absolute_gap,relative_gap_pct
0,pr_auc,0.48265,0.49991,0.01726,3.57506
1,roc_auc,0.85760,0.86909,0.01149,1.33928
2,brier,0.09163,0.09528,0.00365,3.98746
3,log_loss,0.29443,0.29954,0.00511,1.73630


## Point-in-time segment post-hoc analysis

The frozen September segmentation artifact assigns November households from history through November 30. Segment is not an ET-2 input. Aggregate reporting contains no unnecessary household identities.

In [8]:
bundle=load_bundle(ROOT/'models'/'segmentation'/'model.joblib');assignments=assign_from_transactions(bundle,transactions,products,snapshot_date=LOCKBOX_SNAPSHOT);assignments=assignments[assignments.eligibility_status.eq('eligible')][['household_id','segment_code']]
lockbox=lockbox.merge(assignments,on='household_id',how='left',validate='one_to_one');assert lockbox.segment_code.notna().all()
segments=lockbox.groupby('segment_code').agg(households=('household_id','size'),positives=('return_risk_target','sum'),prevalence=('return_risk_target','mean'),median_risk_score=('risk_score','median'),top_10_flagged=('top_10_flag','sum'),within_segment_flag_rate=('top_10_flag','mean')).reset_index();segments['share_of_top10_flags']=segments.top_10_flagged/lockbox.top_10_flag.sum();segments.to_csv(ART/'november_segment_posthoc.csv',index=False);display(segments.round(5))

,segment_code,households,positives,prevalence,median_risk_score,top_10_flagged,within_segment_flag_rate,share_of_top10_flags
0,HIGH_ENGAGEMENT_BROAD,1186,56,0.04722,0.02132,3,0.00253,0.01288
1,LOWER_ENGAGEMENT_FOCUSED,845,256,0.30296,0.22504,219,0.25917,0.93991
2,PROMOTION_BASKET_BUILDERS,291,29,0.09966,0.04883,11,0.03780,0.04721


## Score sanity

In [9]:
sanity=pd.Series({'eligible_score_count':len(lockbox),'minimum':lockbox.risk_score.min(),'p05':lockbox.risk_score.quantile(.05),'p25':lockbox.risk_score.quantile(.25),'median':lockbox.risk_score.median(),'p75':lockbox.risk_score.quantile(.75),'p95':lockbox.risk_score.quantile(.95),'maximum':lockbox.risk_score.max(),'nan_scores':lockbox.risk_score.isna().sum(),'infinite_scores':np.isinf(lockbox.risk_score).sum(),'duplicate_household_ids':lockbox.household_id.duplicated().sum()});assert lockbox.risk_score.between(0,1).all();sanity.to_csv(ART/'november_score_sanity.csv',header=['value']);sanity

eligible_score_count       2322.000000
minimum                       0.002422
p05                           0.005325
p25                           0.016841
median                        0.061312
p75                           0.189978
p95                           0.418659
maximum                       0.637629
nan_scores                    0.000000
infinite_scores               0.000000
duplicate_household_ids       0.000000
dtype: float64

## Final evaluation artifact and lockbox consumption

The final artifact adds only the observed target and post-hoc segment to the already-frozen score/rank/flag rows; it includes no raw transactions.

In [10]:
final_columns=['household_id','snapshot_date','risk_score','risk_rank','top_05_flag','top_10_flag','top_20_flag','return_risk_target','segment_code'];lockbox.loc[:,final_columns].to_csv(ART/'november_lockbox_scores.csv',index=False)
assert PREOUTCOME.exists() and (ART/'november_lockbox_scores.csv').exists()
print('THE NOVEMBER 30 RETURN-RISK LOCKBOX IS NOW CONSUMED.')

THE NOVEMBER 30 RETURN-RISK LOCKBOX IS NOW CONSUMED.


## Generalization assessment and limitations

The predeclared assessment is **STRONG GENERALIZATION**: PR-AUC remains far above prevalence, ET-2 stays competitive with and slightly above the heuristic, top-10% lift is 3.77x, and score sanity is clean. Do not retune after seeing the result. This one-year grocery panel has household dependence, seasonal limitations, and observational—not causal—features.

> **THE NOVEMBER 30 RETURN-RISK LOCKBOX IS NOW CONSUMED.** It can never again be treated as fresh data for feature engineering, model selection, tuning, calibration, capacity/threshold choice, or preprocessing changes.